# No-weather tabular baseline proof of concept

This notebook trains the first reproducible baseline from the immutable, uploadable CSV release for 2026-05-11 through 2026-08-22. It verifies the release checksums, schema, CSV payload digest, and row count *before* loading rows for fitting. It deliberately uses no weather data and only the manifest-provided feature allowlist.

Run it from either the repository root or `notebooks/`. It writes a model bundle, readable contracts, metrics, and a run manifest under `artifacts/`, keyed to the verified release identity.

In [ ]:
from __future__ import annotations

import csv
import gzip
import hashlib
import json
import sys
from pathlib import Path

import pandas as pd


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'wildfire_data').is_dir():
            return candidate
    raise RuntimeError('run this notebook from inside the wildfire-detection repository')


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
SOURCE_ROOT = REPOSITORY_ROOT / 'src'
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

EXPECTED_RELEASE_RELATIVE_PATH = Path(
    'releases/wildfire-spread-firms-feds-no-weather-2026-05-11_to_2026-08-22'
)
RELEASE_DIRECTORY = REPOSITORY_ROOT / EXPECTED_RELEASE_RELATIVE_PATH
CANDIDATE_CSV_NAME = 'candidate_examples.csv.gz'
TARGET_COLUMN = 'target_newly_burned_12h'
ANCHOR_COLUMN = 'anchor_at'
SPLIT_GROUP_COLUMN = 'source_snapshot_time'
EXPECTED_SOURCE_SNAPSHOT_START = '2026-05-11'
EXPECTED_SOURCE_SNAPSHOT_END = '2026-08-22'
NO_WEATHER_STATUS = 'unavailable-no-issued-forecast-features'
NO_WEATHER_POLICY = 'exclude-open-meteo-retrospective-exports/v1'

if not RELEASE_DIRECTORY.is_dir():
    raise FileNotFoundError(
        'expected release directory is missing: ' + str(RELEASE_DIRECTORY)
    )

RELEASE_DIRECTORY


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as source:
        while chunk := source.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_gzip_payload(path: Path) -> str:
    digest = hashlib.sha256()
    with gzip.open(path, 'rb') as source:
        while chunk := source.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def read_checksum_file(release_directory: Path) -> dict[str, str]:
    checksum_path = release_directory / 'SHA256SUMS'
    if not checksum_path.is_file():
        raise FileNotFoundError('release is missing SHA256SUMS')

    entries: dict[str, str] = {}
    for line_number, line in enumerate(
        checksum_path.read_text(encoding='utf-8').splitlines(), start=1
    ):
        digest, separator, relative_name = line.partition('  ')
        if (
            not separator
            or len(digest) != 64
            or any(character not in '0123456789abcdef' for character in digest)
            or not relative_name
        ):
            raise ValueError(f'invalid SHA256SUMS entry on line {line_number}')
        relative_path = Path(relative_name)
        if relative_path.is_absolute() or '..' in relative_path.parts:
            raise ValueError(f'unsafe SHA256SUMS path on line {line_number}')
        if relative_name in entries:
            raise ValueError(f'duplicate SHA256SUMS path: {relative_name}')
        entries[relative_name] = digest

    if not entries or 'SHA256SUMS' in entries:
        raise ValueError('SHA256SUMS must list release files but not itself')
    return entries


def read_json_object(path: Path, label: str) -> dict:
    try:
        document = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError) as error:
        raise ValueError(f'could not read {label}: {path}') from error
    if not isinstance(document, dict):
        raise ValueError(f'{label} must be a JSON object')
    return document


def verify_release(release_directory: Path) -> dict:
    checksum_entries = read_checksum_file(release_directory)
    required_paths = {'dataset_manifest.json', 'schema.json', CANDIDATE_CSV_NAME}
    missing = sorted(required_paths - set(checksum_entries))
    if missing:
        raise ValueError('SHA256SUMS is missing required paths: ' + ', '.join(missing))

    resolved_release = release_directory.resolve()
    for relative_name, expected_digest in checksum_entries.items():
        path = (release_directory / relative_name).resolve()
        try:
            path.relative_to(resolved_release)
        except ValueError as error:
            raise ValueError(f'checksum path escapes release: {relative_name}') from error
        if not path.is_file():
            raise FileNotFoundError(f'checksummed release file is missing: {relative_name}')
        actual_digest = sha256_file(path)
        if actual_digest != expected_digest:
            raise ValueError(f'SHA256 mismatch for {relative_name}')

    manifest_path = release_directory / 'dataset_manifest.json'
    schema_path = release_directory / 'schema.json'
    manifest = read_json_object(manifest_path, 'dataset manifest')
    schema = read_json_object(schema_path, 'release schema')

    if manifest.get('kind') != 'wildfire-spread-candidate-dataset-release':
        raise ValueError('unexpected release manifest kind')
    if manifest.get('source_snapshot_start_date') != EXPECTED_SOURCE_SNAPSHOT_START:
        raise ValueError('release has an unexpected source-snapshot start date')
    if manifest.get('source_snapshot_end_date') != EXPECTED_SOURCE_SNAPSHOT_END:
        raise ValueError('release has an unexpected source-snapshot end date')
    if manifest.get('schema_version') != schema.get('schema_version'):
        raise ValueError('manifest and schema versions disagree')

    formats = schema.get('formats')
    if not isinstance(formats, dict) or not isinstance(formats.get('candidate_examples'), dict):
        raise ValueError('release schema does not define candidate CSV format')
    candidate_format = formats['candidate_examples']
    if candidate_format.get('csv_gzip_path') != CANDIDATE_CSV_NAME:
        raise ValueError('release schema points to an unexpected candidate CSV')

    feature_columns = manifest.get('model_feature_columns')
    if not isinstance(feature_columns, list) or not feature_columns:
        raise ValueError('release manifest lacks a non-empty feature allowlist')
    if any(not isinstance(name, str) or not name for name in feature_columns):
        raise ValueError('feature allowlist contains an invalid field')
    if len(feature_columns) != len(set(feature_columns)):
        raise ValueError('feature allowlist contains duplicate fields')
    if any('weather' in name.lower() for name in feature_columns):
        raise ValueError('no-weather baseline refuses weather model features')
    if schema.get('model_feature_columns') != feature_columns:
        raise ValueError('schema and manifest feature allowlists disagree')
    if schema.get('target_column') != TARGET_COLUMN:
        raise ValueError('release schema has an unexpected target column')

    weather = manifest.get('weather')
    expected_weather = {
        'available': False,
        'status': NO_WEATHER_STATUS,
        'policy': NO_WEATHER_POLICY,
    }
    if weather != expected_weather:
        raise ValueError('release does not satisfy the required no-weather policy')

    csv_columns = manifest.get('candidate_examples_csv_columns')
    if not isinstance(csv_columns, list) or not csv_columns:
        raise ValueError('release manifest lacks candidate CSV columns')
    if candidate_format.get('csv_columns') != csv_columns:
        raise ValueError('schema and manifest candidate CSV columns disagree')
    if len(csv_columns) != len(set(csv_columns)):
        raise ValueError('candidate CSV schema contains duplicate columns')
    if not isinstance(manifest.get('candidate_row_count'), int) or manifest['candidate_row_count'] < 1:
        raise ValueError('candidate row count must be a positive integer')
    if not isinstance(manifest.get('candidate_examples_csv_content_sha256'), str):
        raise ValueError('release manifest lacks candidate CSV payload digest')

    return {
        'manifest': manifest,
        'schema': schema,
        'checksum_entries': checksum_entries,
        'manifest_file_sha256': sha256_file(manifest_path),
        'schema_file_sha256': sha256_file(schema_path),
        'sha256sums_file_sha256': sha256_file(release_directory / 'SHA256SUMS'),
    }


verified_release = verify_release(RELEASE_DIRECTORY)
verified_release['manifest']


In [ ]:
def csv_header_and_row_count(path: Path) -> tuple[list[str], int]:
    with gzip.open(path, 'rt', encoding='utf-8', newline='') as source:
        reader = csv.reader(source)
        try:
            header = next(reader)
        except StopIteration as error:
            raise ValueError('candidate CSV is empty') from error
        if not header or len(header) != len(set(header)):
            raise ValueError('candidate CSV has an invalid header')
        row_count = 0
        for line_number, row in enumerate(reader, start=2):
            if len(row) != len(header):
                raise ValueError(f'candidate CSV has a malformed row at line {line_number}')
            row_count += 1
    return header, row_count


CANDIDATE_CSV_PATH = RELEASE_DIRECTORY / CANDIDATE_CSV_NAME
manifest = verified_release['manifest']
csv_header, csv_row_count = csv_header_and_row_count(CANDIDATE_CSV_PATH)
if csv_header != manifest['candidate_examples_csv_columns']:
    raise ValueError('candidate CSV header does not match the release manifest')
if csv_row_count != manifest['candidate_row_count']:
    raise ValueError('candidate CSV row count does not match the release manifest')

csv_payload_sha256 = sha256_gzip_payload(CANDIDATE_CSV_PATH)
if csv_payload_sha256 != manifest['candidate_examples_csv_content_sha256']:
    raise ValueError('candidate CSV decompressed payload digest does not match the manifest')

release_verification_summary = {
    'release_directory': str(EXPECTED_RELEASE_RELATIVE_PATH),
    'candidate_build_id': manifest['candidate_build_id'],
    'candidate_rows': csv_row_count,
    'candidate_csv_file_sha256': verified_release['checksum_entries'][CANDIDATE_CSV_NAME],
    'candidate_csv_payload_sha256': csv_payload_sha256,
    'weather': manifest['weather'],
}
release_verification_summary


In [ ]:
# This is intentionally the only training-row input. Do not load the JSONL
# companion or unscored positives into this proof-of-concept model.
examples = pd.read_csv(CANDIDATE_CSV_PATH, compression='gzip')

if list(examples.columns) != manifest['candidate_examples_csv_columns']:
    raise ValueError('pandas changed or failed to read the verified candidate CSV header')
if len(examples) != csv_row_count:
    raise ValueError('pandas row count differs from the verified candidate CSV')

required_columns = {
    TARGET_COLUMN,
    'binary_training_eligible',
    'example_id',
    ANCHOR_COLUMN,
    SPLIT_GROUP_COLUMN,
    'weather_available',
    'weather_missing_indicator',
    'weather_feature_status',
    'weather_input_policy',
}
missing_columns = sorted(required_columns - set(examples.columns))
if missing_columns:
    raise ValueError('candidate CSV is missing required columns: ' + ', '.join(missing_columns))

feature_columns = manifest['model_feature_columns']
missing_features = sorted(set(feature_columns) - set(examples.columns))
if missing_features:
    raise ValueError('candidate CSV is missing model features: ' + ', '.join(missing_features))
if any('weather' in name.lower() for name in feature_columns):
    raise ValueError('no-weather baseline refuses weather model features')

def require_binary_values(series: pd.Series, label: str, *, require_both: bool) -> pd.Series:
    try:
        values = pd.to_numeric(series, errors='raise')
    except (TypeError, ValueError) as error:
        raise ValueError(f'{label} must be numeric binary values') from error
    if values.isna().any() or not values.isin([0, 1]).all():
        raise ValueError(f'{label} must contain only 0 and 1 without missing values')
    if require_both and set(values.astype(int).unique()) != {0, 1}:
        raise ValueError(f'{label} must contain both target classes')
    return values.astype('int8')

examples[TARGET_COLUMN] = require_binary_values(
    examples[TARGET_COLUMN], TARGET_COLUMN, require_both=True
)
examples['binary_training_eligible'] = require_binary_values(
    examples['binary_training_eligible'], 'binary_training_eligible', require_both=False
)
if not (examples['binary_training_eligible'] == 1).all():
    raise ValueError('candidate CSV contains rows that are not eligible for binary training')

if examples['example_id'].isna().any() or examples['example_id'].astype(str).str.strip().eq('').any():
    raise ValueError('candidate CSV contains a missing or blank example_id')
if not examples['example_id'].is_unique:
    raise ValueError('candidate CSV contains duplicate example_id values')

for time_column in (ANCHOR_COLUMN, SPLIT_GROUP_COLUMN):
    parsed = pd.to_datetime(examples[time_column], utc=True, errors='coerce')
    if parsed.isna().any():
        raise ValueError(f'candidate CSV has an invalid UTC timestamp in {time_column}')

weather_available = require_binary_values(
    examples['weather_available'], 'weather_available', require_both=False
)
weather_missing = require_binary_values(
    examples['weather_missing_indicator'], 'weather_missing_indicator', require_both=False
)
if not (weather_available == 0).all() or not (weather_missing == 1).all():
    raise ValueError('candidate CSV violates the no-weather row policy')
if not examples['weather_feature_status'].eq(NO_WEATHER_STATUS).all():
    raise ValueError('candidate CSV has an unexpected weather feature status')
if not examples['weather_input_policy'].eq(NO_WEATHER_POLICY).all():
    raise ValueError('candidate CSV has an unexpected weather input policy')

for feature_name in feature_columns:
    try:
        examples[feature_name] = pd.to_numeric(examples[feature_name], errors='raise')
    except (TypeError, ValueError) as error:
        raise ValueError(f'model feature is not numeric: {feature_name}') from error

examples.loc[:, [TARGET_COLUMN, 'binary_training_eligible', *feature_columns]].describe(include='all')


In [ ]:
from wildfire_data.tabular_baseline import persist_tabular_baseline, train_tabular_baseline

baseline_result = train_tabular_baseline(
    examples,
    target_column=TARGET_COLUMN,
    feature_columns=feature_columns,
    anchor_column=ANCHOR_COLUMN,
    split_group_column=SPLIT_GROUP_COLUMN,
    random_state=0,
)

baseline_metrics = baseline_result.metrics.as_dict()
{
    'metrics': {
        key: value
        for key, value in baseline_metrics.items()
        if key != 'calibration_bins'
    },
    'feature_contract': baseline_result.feature_contract.as_dict(),
}


In [ ]:
release_identity = verified_release['manifest_file_sha256'][:16]
artifact_directory = REPOSITORY_ROOT / 'artifacts' / f'tabular-baseline-{release_identity}'
persisted = persist_tabular_baseline(
    baseline_result,
    artifact_directory,
    basename='no_weather_candidate_baseline',
)

def repository_relative(path: Path) -> str:
    return path.resolve().relative_to(REPOSITORY_ROOT).as_posix()


run_manifest = {
    'schema_version': 1,
    'kind': 'wildfire-no-weather-tabular-baseline-run',
    'release': {
        'directory': EXPECTED_RELEASE_RELATIVE_PATH.as_posix(),
        'candidate_build_id': manifest['candidate_build_id'],
        'source_snapshot_start_date': manifest['source_snapshot_start_date'],
        'source_snapshot_end_date': manifest['source_snapshot_end_date'],
        'dataset_manifest_file_sha256': verified_release['manifest_file_sha256'],
        'schema_file_sha256': verified_release['schema_file_sha256'],
        'sha256sums_file_sha256': verified_release['sha256sums_file_sha256'],
        'candidate_examples_csv_file_sha256': verified_release['checksum_entries'][CANDIDATE_CSV_NAME],
        'candidate_examples_csv_content_sha256': csv_payload_sha256,
        'candidate_row_count': csv_row_count,
        'weather': manifest['weather'],
    },
    'training': {
        'target_column': TARGET_COLUMN,
        'feature_columns': feature_columns,
        'anchor_column': ANCHOR_COLUMN,
        'split_group_column': SPLIT_GROUP_COLUMN,
        'random_state': 0,
    },
    'feature_contract': baseline_result.feature_contract.as_dict(),
    'evaluation_metrics': baseline_metrics,
    'artifacts': {
        'model_bundle': repository_relative(persisted.model_bundle_path),
        'feature_contract': repository_relative(persisted.feature_contract_path),
        'metrics': repository_relative(persisted.metrics_path),
    },
}
run_manifest_path = artifact_directory / 'run_manifest.json'
temporary_run_manifest_path = artifact_directory / '.run_manifest.json.tmp'
temporary_run_manifest_path.write_text(
    json.dumps(run_manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8'
)
temporary_run_manifest_path.replace(run_manifest_path)

{
    'artifact_directory': repository_relative(artifact_directory),
    'run_manifest': repository_relative(run_manifest_path),
    'metrics': {
        key: value
        for key, value in baseline_metrics.items()
        if key != 'calibration_bins'
    },
}


The holdout metrics are for a later chronological group of `source_snapshot_time` values; they are not an incident-held-out or region-held-out estimate. Target=0 remains a FIRMS-seeded weak-negative proxy, not observed clear/no-burn.